# 04. Stage 2: candidate Ranking (LightGBM Ranker)

Notebook này xây dựng mô hình chấm điểm chi tiết (Ranking) để chọn ra các bộ phim tốt nhất từ danh sách ứng viên thô được tạo ra từ giai đoạn Retrieval.

---

### Phân tích Quyết định Thiết kế:
*   **Tại sao chọn LightGBM LambdaRank?**
    *   LightGBM thuộc nhóm Gradient Boosting Decision Trees (GBDT) - là giải thuật chuẩn công nghiệp tốt nhất cho dữ liệu cấu trúc bảng (tabular). Biến thể **LambdaRank** tối ưu hóa trực tiếp hàm mục tiêu NDCG (thay vì tối ưu MSE/BCE đơn thuần), giúp xếp hạng các phim được yêu thích thực sự lên đầu danh sách hiệu quả hơn. Thuật toán phân tách dựa trên histogram giúp tốc độ train cực nhanh trên CPU.
*   **Tại sao không chọn Logistic Regression?**
    *   Hồi quy tuyến tính hoặc Logistic chỉ học được các mối quan hệ tuyến tính giữa các đặc trưng, trừ khi lập trình viên tự thiết kế các thuộc tính chéo (feature crosses) một cách thủ công và phức tạp. GBDT tự động phát hiện các mối quan hệ tương tác phi tuyến và giao cắt đặc trưng thông qua các nhánh quyết định của cây.
*   **Tại sao không chọn DeepFM hay NeuMF (Deep Learning) làm Ranker chính?**
    *   Chúng ta đang xây dựng kiến trúc thuần ML. Ngoài ra, Deep Learning cần tài nguyên tính toán lớn (GPU), thời gian huấn luyện lâu hơn gấp 10 lần, và rất dễ bị quá khớp (overfit) khi tập dữ liệu huấn luyện tương đối nhỏ (~2,000 ratings).


### Bước 1: Khởi tạo và Tải dữ liệu giai đoạn trước
Tế bào này tải tập dữ liệu train ratings, thông tin chi tiết phim và người dùng, các dict mapping ID, cùng các mô hình Retrieval (ALS và BM25) đã huấn luyện thành công ở Stage 1 để sử dụng cho việc trích xuất đặc trưng chéo.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import pickle
import lightgbm as lgb
import scipy.sparse as sp
from sklearn.model_selection import train_test_split

# Thêm đường dẫn cha để import recsys_utils
sys.path.append(os.path.abspath('..'))
from recsys_utils import BM25

# Load dữ liệu đã xử lý
train_ratings = pd.read_csv("processed_data/train_ratings.csv")
movies_df = pd.read_csv(os.path.join("..", "..", "data", "crawler", "movies_crawled.csv"))
users_df = pd.read_csv(os.path.join("..", "..", "data", "simulator", "sim_users.csv"))

with open("processed_data/id_mappings.pkl", "rb") as f:
    user_to_idx, movie_to_idx, idx_to_movie = pickle.load(f)
    
with open("processed_data/user_interacted_items.pkl", "rb") as f:
    user_interacted_items = pickle.load(f)

# Load retrieval models/matrices for feature engineering
with open("models/als_model.pkl", "rb") as f:
    als_model = pickle.load(f)
    
with open("models/bm25_model.pkl", "rb") as f:
    bm25 = pickle.load(f)

with open("processed_data/user_item_matrix.pkl", "rb") as f:
    user_item_matrix = pickle.load(f)


### Bước 2: Sinh Tập dữ liệu Huấn luyện Xếp hạng (Negative Sampling)
Để huấn luyện mô hình xếp hạng nhị phân Stage 2, ta cần cả mẫu dương (positive) và mẫu âm (negative):
1. **Mẫu dương (Label = 1)**: Là các cặp User-Movie có rating thực tế $\ge 3.5$ trong tập train.
2. **Mẫu âm (Label = 0)**: Dùng chiến lược **Hard Negative Sampling kết hợp Random Negatives** (Chuẩn công nghiệp):
   * Với mỗi user, ta chạy mô hình iALS để sinh 200 đề xuất tốt nhất. Các phim trong top này mà user **chưa tương tác** được gọi là **Hard Negatives** (phim gần đúng gu nhưng user không chọn).
   * Lấy mẫu 50% Hard Negatives và 50% Random Negatives để giúp Ranker phân biệt tối ưu giữa phim liên quan gần và phim hoàn toàn không liên quan.


In [ ]:
# 1. Tạo tập dữ liệu huấn luyện cho Ranker (Hard + Random Negatives)
np.random.seed(42)

# Precompute iALS candidates cho mọi user trong train ratings phục vụ trích xuất hard negatives
print("Đang sinh danh sách hard negatives từ iALS...")
als_hard_candidates = {}
for u in train_ratings['userId'].unique():
    u_idx = user_to_idx.get(u, None)
    if u_idx is not None:
        ids, _ = als_model.recommend(u_idx, user_item_matrix[u_idx], N=200)
        movie_ids = [idx_to_movie[i] for i in ids if i in idx_to_movie]
        als_hard_candidates[u] = movie_ids

ranking_data = []
all_movie_ids = list(movie_to_idx.keys())

for _, row in train_ratings.iterrows():
    u = int(row['userId'])
    pos_item = int(row['movieId'])
    rating = row['rating']
    
    label = 1 if rating >= 3.5 else 0
    ranking_data.append({'userId': u, 'movieId': pos_item, 'label': label})
    
    interacted = user_interacted_items.get(u, set())
    hard_negs = als_hard_candidates.get(u, [])
    # Lọc bỏ các phim user đã tương tác
    hard_negs_clean = [m for m in hard_negs if m not in interacted]
    
    # Lấy mẫu tối đa 2 hard negatives
    num_hard_to_sample = min(2, len(hard_negs_clean))
    sampled_negs = []
    if num_hard_to_sample > 0:
        sampled_negs = list(np.random.choice(hard_negs_clean, size=num_hard_to_sample, replace=False))
        
    # Phần còn lại bù bằng random negatives
    num_rand_to_sample = 4 - len(sampled_negs)
    for _ in range(num_rand_to_sample):
        neg_item = np.random.choice(all_movie_ids)
        while neg_item in interacted or neg_item in sampled_negs:
            neg_item = np.random.choice(all_movie_ids)
        sampled_negs.append(neg_item)
        
    for neg_item in sampled_negs:
        ranking_data.append({'userId': u, 'movieId': neg_item, 'label': 0})

df_rank = pd.DataFrame(ranking_data)
print(f"Tổng số dòng huấn luyện ranker: {len(df_rank)}")
print("Phân phối nhãn:")
print(df_rank['label'].value_counts())


### Bước 3: Thiết kế Đặc trưng (Feature Engineering)
Để mô hình học máy đạt độ chính xác cao, ta thiết kế 3 nhóm đặc trưng chính cho mỗi cặp User-Movie:
1. **Đặc trưng của Phim (Movie Features)**:
   - `popularity`: Độ phổ biến của phim cào từ TMDB.
   - `vote_average`: Điểm đánh giá trung bình toàn cầu.
   - `release_year`: Năm phát hành của phim (trích xuất từ release_date).
2. **Đặc trưng của Người dùng (User Features)**:
   - `user_activity`: Mức độ hoạt động của người dùng (tổng số click/rating).
   - `user_bias`: Xu hướng chấm điểm của người dùng (chênh lệch trung bình so với điểm chung).
3. **Đặc trưng Tương tác chéo (Cross/Affinity Features)**:
   - `genre_overlap`: Số thể loại trùng khớp giữa gu yêu thích của người dùng và thể loại của bộ phim.
   - `als_score`: Điểm số dự đoán từ mô hình Collaborative Filtering (ALS) ở Stage 1.
   - `cb_score`: Điểm số dự đoán độ tương đồng nội dung BM25 từ mô hình Content-Based ở Stage 1.


In [ ]:
# 2. Xây dựng các đặc trưng (Feature Engineering)
movies_df['genres'] = movies_df['genres'].fillna('')
movies_df['director'] = movies_df['director'].fillna('')
movies_df['cast'] = movies_df['cast'].fillna('')
movies_df['keywords'] = movies_df['keywords'].fillna('')

def build_metadata_soup(row):
    genres = row['genres'].replace('|', ' ')
    cast = ' '.join(row['cast'].split('|')[:5])
    keywords = row['keywords'].replace('|', ' ')
    director = row['director'].replace(' ', '')
    return f"{genres} {director} {cast} {keywords}"

movies_df['soup'] = movies_df.apply(build_metadata_soup, axis=1)
movies_unindexed = movies_df.set_index('movieId', drop=False)
movies_df = movies_df.set_index('movieId')
users_df = users_df.set_index('user_id')

# Precompute user liked movie IDs to avoid slow dataframe scans in the loop
train_pos_ratings = train_ratings[train_ratings['rating'] >= 3.5]
user_liked_movies_dict = train_pos_ratings.groupby('userId')['movieId'].apply(list).to_dict()

# Precompute user favorite genres from their actual watched history (rating >= 3.5) with fallback to seed list
user_fav_genres_dict = {}
for uid, row in users_df.iterrows():
    liked_mids = user_liked_movies_dict.get(uid, [])
    genres = set()
    for lmid in liked_mids:
        if lmid in movies_df.index:
            g_str = movies_df.loc[lmid, "genres"]
            if pd.notna(g_str):
                genres.update(str(g_str).split("|"))
                
    if not genres:
        fav_m_str = str(row.get("favorite_movies", ""))
        fav_m_list = [int(m) for m in fav_m_str.split("|") if str(m).isdigit()]
        for fid in fav_m_list:
            if fid in movies_df.index:
                g_str = movies_df.loc[fid, "genres"]
                if pd.notna(g_str):
                    genres.update(str(g_str).split("|"))
    user_fav_genres_dict[uid] = genres

# Precompute dictionary mappings for O(1) loop lookups (bypasses slow pandas .loc)
movie_popularity_dict = movies_df['popularity'].to_dict()
movie_vote_average_dict = movies_df['vote_average'].to_dict()
movie_genres_dict = movies_df['genres'].to_dict()
movie_genres_sets = {mid: set(str(g).split('|')) for mid, g in movie_genres_dict.items()}
movie_release_date_dict = movies_df['release_date'].to_dict()
movies_soup_dict = movies_unindexed['soup'].to_dict()

user_activity_dict = users_df['activity_level'].to_dict()
user_bias_dict = users_df['user_bias'].to_dict()

# Đảm bảo df_rank được sắp xếp theo userId để tối ưu việc cache profile similarity
df_rank_sorted_by_user = df_rank.sort_values(by='userId')

uids = df_rank_sorted_by_user['userId'].values
mids = df_rank_sorted_by_user['movieId'].values

# Vectorized/fast lookup arrays
popularity_vec = [movie_popularity_dict.get(mid, 1.0) for mid in mids]
vote_average_vec = [movie_vote_average_dict.get(mid, 5.0) for mid in mids]
user_activity_vec = [user_activity_dict.get(uid, 15) for uid in uids]
user_bias_vec = [user_bias_dict.get(uid, 0.0) for uid in uids]

genre_overlaps = []
for uid, mid in zip(uids, mids):
    user_favs = user_fav_genres_dict.get(uid, set())
    m_genres = movie_genres_sets.get(mid, set())
    genre_overlaps.append(len(user_favs.intersection(m_genres)))

release_years = []
for mid in mids:
    try:
        release_years.append(int(str(movie_release_date_dict.get(mid, "2010"))[:4]))
    except:
        release_years.append(2010)

# Vectorized ALS scores (avoid loop row-by-row matrix multiplication)
u_idx_arr = np.array([user_to_idx.get(u, -1) for u in uids])
m_idx_arr = np.array([movie_to_idx.get(m, -1) for m in mids])
valid_mask = (u_idx_arr != -1) & (m_idx_arr != -1)
als_scores = np.zeros(len(df_rank_sorted_by_user))
if valid_mask.any():
    als_scores[valid_mask] = np.sum(
        als_model.user_factors[u_idx_arr[valid_mask]] * als_model.item_factors[m_idx_arr[valid_mask]], 
        axis=1
    )

# Fast CB scores using precomputed BoW vectors (avoid text tokenization and prevent leakage)
user_query_bow_dict = {}
for uid, liked_movies in user_liked_movies_dict.items():
    liked_ids_clean = [lid for lid in liked_movies if lid in movies_soup_dict]
    liked_ids_clean = liked_ids_clean[-20:]
    if liked_ids_clean:
        liked_idxs = [movie_to_idx[lid] for lid in liked_ids_clean if lid in movie_to_idx]
        if liked_idxs:
            user_query_bow_dict[uid] = bm25.tf[liked_idxs].sum(axis=0)

# Precompute BM25 scores vector for all users instantly
user_bm25_all_dict = {}
for uid, u_bow in user_query_bow_dict.items():
    if u_bow is not None:
        user_bm25_all_dict[uid] = bm25.transform_from_bow(u_bow)

cb_scores = []
for uid, mid in zip(uids, mids):
    m_idx = movie_to_idx.get(mid, None)
    u_scores = user_bm25_all_dict.get(uid, None)
    cb_score = u_scores[m_idx] if (u_scores is not None and m_idx is not None) else 0.0
    cb_scores.append(cb_score)

# Instantly build features DataFrame
X = pd.DataFrame({
    'popularity': popularity_vec,
    'vote_average': vote_average_vec,
    'genre_overlap': genre_overlaps,
    'release_year': release_years,
    'user_activity': user_activity_vec,
    'user_bias': user_bias_vec,
    'als_score': als_scores,
    'cb_score': cb_scores
})
y = df_rank_sorted_by_user['label']

df_rank_sorted_by_user['group_key'] = df_rank_sorted_by_user['userId']
X_sorted = X
y_sorted = y
groups = df_rank_sorted_by_user.groupby('group_key', sort=False).size().values

print(f"Features head:")
display(X_sorted.head(3))


### Bước 4: Huấn luyện mô hình xếp hạng LightGBM (LambdaRank)

#### Nguyên lý thuật toán LambdaRank:
Khác với mô hình phân loại nhị phân thông thường tối ưu hàm Binary Cross Entropy độc lập cho từng dòng, LambdaRank là giải thuật xếp hạng dạng cặp (**Pairwise Learning-to-Rank**). 
Nó định nghĩa một đạo hàm giả (gọi là $\lambda$) cho mỗi cặp phim $(i, j)$ được đề xuất cho cùng một người dùng:
$$\lambda_{ij} = \frac{\partial C_{ij}}{\partial s_i} = -\frac{1}{1 + e^{s_i - s_j}} \cdot |\Delta \text{NDCG}|$$
Trong đó:
*   $s_i$ và $s_j$ lần lượt là điểm số dự đoán của mô hình cho phim tốt $i$ và phim kém hơn $j$.
*   $|\Delta \text{NDCG}|$ là lượng thay đổi của chỉ số xếp hạng toàn cục NDCG nếu ta hoán đổi vị trí của phim $i$ và $j$ trong danh sách đề xuất. 
*   Cơ chế này giúp mô hình tập trung tối ưu hóa thứ tự xếp hạng của các phim ở đầu danh sách (vị trí quan trọng nhất) thay vì tối ưu hóa toàn bộ dữ liệu một cách cào bằng.

#### So sánh các cách tiếp cận Learning-to-Rank (LTR):
| Cách tiếp cận | Hàm mục tiêu | Ưu điểm | Nhược điểm |
| :--- | :--- | :--- | :--- |
| **Pointwise** | Hồi quy hoặc Phân loại nhị phân độc lập từng dòng (BCE/MSE) | Đơn giản, dùng được các thư viện phân loại truyền thống. | Coi các phim độc lập, không học được thứ tự tương đối giữa các phim trong cùng danh sách. |
| **Pairwise** (LambdaRank) | Tối ưu hóa thứ tự cặp phim, nhân với trọng số ảnh hưởng NDCG | Hiệu năng xếp hạng thực tế rất cao, tập trung vào top đầu danh sách. | Cấu trúc dữ liệu huấn luyện phức tạp hơn (cần định nghĩa `group` danh sách). |
| **Listwise** | Tối ưu hóa trực tiếp hàm phân phối xác suất của toàn bộ danh sách | Về mặt lý thuyết là tối ưu nhất. | Rất khó cài đặt, chi phí tính toán cực kỳ lớn. |

Tế bào này khởi tạo mô hình `LGBMRanker` với mục tiêu `objective='lambdarank'`, nhóm dữ liệu theo từng User (`group=groups`) và huấn luyện mô hình.


In [ ]:
# 3. Huấn luyện LightGCN Ranker (LambdaRank)
ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    eval_at=[10],
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

ranker.fit(
    X_sorted, y_sorted,
    group=groups
)

# Lưu Ranker model
with open("models/lgb_ranker.pkl", "wb") as f:
    pickle.dump(ranker, f)
    
print("Huấn luyện thành công LGBMRanker!")
